# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# MaskCut pseudo-masks as an additional CNN channel

This notebook implements the central workflow in `Report-199_Spring.pdf` on Kather-2016. MaskCut masks are generated once and cached. The same case-grouped split and CNN are then used to compare:

1. Original RGB tiles (`rgb`)
2. RGB pixels retained only inside the selected MaskCut region (`masked_rgb`)
3. RGB plus the selected binary MaskCut mask as a fourth channel (`rgb_mask`)

The report used the reverse (complement) of the raw MaskCut output. `INVERT_MASK = True` below reproduces that convention without regenerating the cached masks. The filenames identify ten source cases, and all splits are grouped by source case to prevent tiles from one case appearing in multiple splits.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader

sns.set_theme(style="whitegrid")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT
    / "Colorectal Histology MNIST"
    / "Kather_texture_2016_image_tiles_5000"
    / "Kather_texture_2016_image_tiles_5000"
)
MASKCUT_DIR = PROJECT_ROOT / "maskcut"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "baseline_cnn"
MASK_DIR = ARTIFACT_DIR / "masks"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
PLOT_DIR = ARTIFACT_DIR / "plots"
for directory in (ARTIFACT_DIR, MASK_DIR, CHECKPOINT_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Artifacts:", ARTIFACT_DIR)

In [ ]:
from Methods.BaselineCNN import (
    BaselineCNN,
    HistologyTileDataset,
    discover_images,
    evaluate_model,
    fit_model,
    make_grouped_split,
    set_seed,
)
from Methods.BaselineCNN.maskcut_preprocess import generate_pseudo_masks

SEED = 41
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 40
LEARNING_RATE = 1e-3
INVERT_MASK = True  # Report convention: use 1 - raw MaskCut mask.
MASK_VARIANT = "reverse_mask" if INVERT_MASK else "raw_mask"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
DEVICE

## 1. Discover images and create a case-grouped split

This is intentionally stricter than a random tile split. It evaluates whether the model transfers to unseen source cases instead of recognizing case-specific stain and acquisition patterns.

In [ ]:
manifest = discover_images(DATASET_DIR)
manifest = make_grouped_split(manifest, random_state=SEED)
manifest.to_csv(ARTIFACT_DIR / "split_manifest_without_masks.csv", index=False)

class_table = (
    manifest[["class_name", "label"]]
    .drop_duplicates()
    .sort_values("label")
)
CLASS_NAMES = class_table["class_name"].tolist()

print("Images:", len(manifest))
print("Cases:", sorted(manifest["case_id"].unique()))
display(pd.crosstab(manifest["split"], manifest["class_name"]))
display(manifest.groupby("split")["case_id"].unique().to_frame())

## 2. Generate and cache MaskCut pseudo-masks

The first run downloads the DINO weights and processes all 5,000 images. Subsequent runs skip existing masks in `artifacts/baseline_cnn/masks/`. Set `MASK_LIMIT` to a small number such as `16` for a pipeline smoke test, then restore it to `None` for the full experiment.

`fixed_size=160` keeps the computation near the original 150 x 150 resolution and is divisible by the DINO patch size. CRF is disabled initially so the baseline does not depend on an extra compiled package. The PNG files always retain the raw MaskCut output; reversal happens later in the dataset loader.

In [ ]:
RUN_MASK_GENERATION = True
MASK_LIMIT = None  # Use 16 for a smoke test; None processes all images.

if RUN_MASK_GENERATION:
    manifest, mask_failures = generate_pseudo_masks(
        manifest,
        maskcut_dir=MASKCUT_DIR,
        mask_root=MASK_DIR,
        architecture="small",
        patch_size=8,
        tau=0.15,
        number_of_masks=1,
        fixed_size=160,
        use_crf=False,
        overwrite=False,
        device=DEVICE,
        limit=MASK_LIMIT,
    )
    if mask_failures:
        failure_frame = pd.DataFrame(mask_failures)
        failure_frame.to_csv(ARTIFACT_DIR / "mask_failures.csv", index=False)
        display(failure_frame.head())
        print(f"Mask failures: {len(failure_frame)}")
else:
    manifest["mask_path"] = manifest.apply(
        lambda row: str(
            MASK_DIR
            / row["class_name"]
            / f"{Path(row['image_path']).stem}_mask.png"
        ),
        axis=1,
    )

manifest.to_csv(ARTIFACT_DIR / "split_manifest.csv", index=False)
missing_masks = manifest[~manifest["mask_path"].map(lambda value: Path(value).is_file())]
print("Available masks:", len(manifest) - len(missing_masks), "/", len(manifest))
if MASK_LIMIT is None and len(missing_masks):
    raise RuntimeError("Some masks are missing. Inspect artifacts/baseline_cnn/mask_failures.csv")

In [ ]:
from PIL import Image

available = manifest[manifest["mask_path"].map(lambda value: Path(value).is_file())]
examples = available.groupby("class_name", group_keys=False).head(1)
fig, axes = plt.subplots(len(examples), 4, figsize=(12, 2.5 * len(examples)))
for row_index, (_, row) in enumerate(examples.iterrows()):
    image = np.asarray(Image.open(row["image_path"]).convert("RGB"))
    raw_mask = np.asarray(Image.open(row["mask_path"]).convert("L")) > 127
    reverse_mask = np.logical_not(raw_mask)
    overlay = image.copy()
    overlay[reverse_mask] = (
        0.55 * overlay[reverse_mask] + 0.45 * np.array([0, 255, 80])
    ).astype(np.uint8)
    for axis, content, title in zip(
        axes[row_index],
        (image, raw_mask, reverse_mask, overlay),
        (row["class_name"], "Raw MaskCut mask", "Reverse mask", "Reverse-mask overlay"),
    ):
        axis.imshow(content, cmap="gray" if content.ndim == 2 else None)
        axis.set_title(title)
        axis.axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR / "mask_examples_reverse_mask.png", dpi=180, bbox_inches="tight")
plt.show()

## 3. Train the report-style comparison

All three conditions use the same split, architecture, optimizer, and early-stopping rule. With `INVERT_MASK = True`, both mask-based conditions consume the reverse mask. Output names include `reverse_mask`, preserving results from earlier raw-mask runs.

In [ ]:
def label_input_mode(input_mode):
    return input_mode if input_mode == "rgb" else f"{input_mode}_{MASK_VARIANT}"


def make_loaders(input_mode, source_manifest=manifest, batch_size=BATCH_SIZE):
    loaders = {}
    for split in ("train", "validation", "test"):
        split_frame = source_manifest[source_manifest["split"] == split]
        dataset = HistologyTileDataset(
            split_frame,
            input_mode=input_mode,
            augment=split == "train",
            invert_mask=INVERT_MASK,
        )
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=split == "train",
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=NUM_WORKERS > 0,
        )
    return loaders


RUN_TRAINING = True
INPUT_MODES = ("rgb", "masked_rgb", "rgb_mask")
histories = {}
evaluations = {}
trained_models = {}

if RUN_TRAINING:
    for input_mode in INPUT_MODES:
        experiment_name = label_input_mode(input_mode)
        print(f"\nTraining: {experiment_name}")
        set_seed(SEED)
        loaders = make_loaders(input_mode)
        model = BaselineCNN(
            in_channels=4 if input_mode == "rgb_mask" else 3,
            num_classes=len(CLASS_NAMES),
        )
        checkpoint_path = CHECKPOINT_DIR / f"baseline_{experiment_name}.pt"
        history = fit_model(
            model,
            loaders["train"],
            loaders["validation"],
            DEVICE,
            checkpoint_path,
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
        )
        evaluation = evaluate_model(
            model,
            loaders["test"],
            DEVICE,
            CLASS_NAMES,
        )
        history.to_csv(ARTIFACT_DIR / f"history_{experiment_name}.csv", index=False)
        histories[experiment_name] = history
        evaluations[experiment_name] = evaluation
        trained_models[experiment_name] = model
        print(
            f"Test accuracy={evaluation['accuracy']:.4f}, "
            f"macro-F1={evaluation['macro_f1']:.4f}"
        )

## 4. Compare performance and learning curves

In [ ]:
summary = pd.DataFrame(
    [
        {
            "input_mode": mode,
            "accuracy": values["accuracy"],
            "balanced_accuracy": values["balanced_accuracy"],
            "macro_f1": values["macro_f1"],
            "test_loss": values["loss"],
        }
        for mode, values in evaluations.items()
    ]
).sort_values("macro_f1", ascending=False)
summary.to_csv(ARTIFACT_DIR / f"test_summary_{MASK_VARIANT}.csv", index=False)
display(summary.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode, history in histories.items():
    axes[0].plot(history["epoch"], history["train_loss"], label=f"{mode}: train")
    axes[0].plot(history["epoch"], history["validation_loss"], linestyle="--", label=f"{mode}: val")
    axes[1].plot(history["epoch"], history["validation_macro_f1"], label=mode)
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
for axis in axes:
    axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / f"learning_curves_{MASK_VARIANT}.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
figure, axes = plt.subplots(1, len(evaluations), figsize=(6 * len(evaluations), 5))
if len(evaluations) == 1:
    axes = [axes]
for axis, (mode, values) in zip(axes, evaluations.items()):
    sns.heatmap(
        values["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )
    axis.set(title=mode, xlabel="Predicted", ylabel="True")
    axis.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.savefig(PLOT_DIR / f"confusion_matrices_{MASK_VARIANT}.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Optional report extensions

The report also investigated training-set sizes of 500, 1,000, and 1,500 images. The next cell repeats that ablation using balanced subsets of the grouped training split. It is disabled by default because it trains six additional models.

In [ ]:
RUN_DATASET_SIZE_ABLATION = False
DATASET_SIZES = (500, 1000, 1500)
size_results = []

if RUN_DATASET_SIZE_ABLATION:
    full_train = manifest[manifest["split"] == "train"]
    non_train = manifest[manifest["split"] != "train"]
    for sample_count in DATASET_SIZES:
        per_class = sample_count // len(CLASS_NAMES)
        sampled_train = (
            full_train.groupby("label", group_keys=False)
            .apply(lambda group: group.sample(min(per_class, len(group)), random_state=SEED))
            .reset_index(drop=True)
        )
        ablation_manifest = pd.concat((sampled_train, non_train), ignore_index=True)
        for input_mode in ("rgb", "rgb_mask"):
            experiment_name = label_input_mode(input_mode)
            set_seed(SEED)
            loaders = make_loaders(input_mode, source_manifest=ablation_manifest)
            model = BaselineCNN(
                in_channels=4 if input_mode == "rgb_mask" else 3,
                num_classes=len(CLASS_NAMES),
            )
            history = fit_model(
                model,
                loaders["train"],
                loaders["validation"],
                DEVICE,
                CHECKPOINT_DIR / f"size_{sample_count}_{experiment_name}.pt",
                epochs=30,
                learning_rate=LEARNING_RATE,
            )
            metrics = evaluate_model(model, loaders["test"], DEVICE, CLASS_NAMES)
            size_results.append(
                {
                    "requested_train_size": sample_count,
                    "actual_train_size": len(sampled_train),
                    "input_mode": experiment_name,
                    "accuracy": metrics["accuracy"],
                    "macro_f1": metrics["macro_f1"],
                }
            )
    size_results = pd.DataFrame(size_results)
    size_results.to_csv(ARTIFACT_DIR / f"dataset_size_ablation_{MASK_VARIANT}.csv", index=False)
    display(size_results)

The final optional trial mirrors the report's ImageNet-pretrained ResNet-18 experiment. For four-channel input, the new mask-channel kernel is initialized from the mean of the pretrained RGB kernels.

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18


def build_resnet18(input_mode):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    if input_mode == "rgb_mask":
        old_projection = model.conv1
        new_projection = nn.Conv2d(
            4,
            old_projection.out_channels,
            kernel_size=old_projection.kernel_size,
            stride=old_projection.stride,
            padding=old_projection.padding,
            bias=False,
        )
        with torch.no_grad():
            new_projection.weight[:, :3] = old_projection.weight
            new_projection.weight[:, 3:4] = old_projection.weight.mean(dim=1, keepdim=True)
        model.conv1 = new_projection
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    return model


RUN_RESNET18 = False
resnet_results = []
if RUN_RESNET18:
    for input_mode in ("rgb", "masked_rgb", "rgb_mask"):
        experiment_name = label_input_mode(input_mode)
        set_seed(SEED)
        loaders = make_loaders(input_mode, batch_size=32)
        model = build_resnet18(input_mode)
        fit_model(
            model,
            loaders["train"],
            loaders["validation"],
            DEVICE,
            CHECKPOINT_DIR / f"resnet18_{experiment_name}.pt",
            epochs=20,
            learning_rate=1e-4,
        )
        metrics = evaluate_model(model, loaders["test"], DEVICE, CLASS_NAMES)
        resnet_results.append(
            {
                "input_mode": experiment_name,
                "accuracy": metrics["accuracy"],
                "balanced_accuracy": metrics["balanced_accuracy"],
                "macro_f1": metrics["macro_f1"],
            }
        )
    resnet_results = pd.DataFrame(resnet_results)
    resnet_results.to_csv(ARTIFACT_DIR / f"resnet18_summary_{MASK_VARIANT}.csv", index=False)
    display(resnet_results)

## 6. Multi-seed significance analysis

This experiment repeats the primary `rgb` versus `rgb_mask_reverse_mask` comparison with ten training seeds while holding the case-grouped split fixed. Results are saved after every model, so an interrupted run can resume without discarding completed seeds.

The pre-specified primary endpoint is test accuracy. Because the two models are evaluated for the same training seeds and test split, the **one-sided paired Wilcoxon signed-rank test** is the primary statistical test. The requested **Wilcoxon rank-sum test** is reported as a secondary analysis, but it ignores the pairing and is therefore less appropriate here. Statistical significance is reported only when the computed p-value is below `0.05`; it is not assumed in advance.

This seed-level analysis measures robustness to training randomness conditional on one fixed test split. It does not make the ten runs equivalent to ten independent patient cohorts. For a manuscript-level generalization claim, repeat the comparison across grouped outer folds or independent cohorts.

In [ ]:
MULTI_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
MULTI_SEED_MODES = ("rgb", "rgb_mask")
RUN_MULTI_SEED = True
MULTI_SEED_RESULTS_PATH = ARTIFACT_DIR / f"multiseed_results_{MASK_VARIANT}.csv"

result_columns = [
    "seed",
    "input_mode",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "test_loss",
]
if MULTI_SEED_RESULTS_PATH.is_file():
    multi_seed_results = pd.read_csv(MULTI_SEED_RESULTS_PATH)
else:
    multi_seed_results = pd.DataFrame(columns=result_columns)

completed_runs = {
    (int(row.seed), row.input_mode)
    for row in multi_seed_results.itertuples(index=False)
}

if RUN_MULTI_SEED:
    for training_seed in MULTI_SEEDS:
        for input_mode in MULTI_SEED_MODES:
            experiment_name = label_input_mode(input_mode)
            run_key = (training_seed, experiment_name)
            if run_key in completed_runs:
                print(f"Skipping completed run: seed={training_seed}, mode={experiment_name}")
                continue

            print(f"\nSeed={training_seed}, mode={experiment_name}")
            set_seed(training_seed)
            loaders = make_loaders(input_mode)
            model = BaselineCNN(
                in_channels=4 if input_mode == "rgb_mask" else 3,
                num_classes=len(CLASS_NAMES),
            )
            checkpoint_path = (
                CHECKPOINT_DIR / "multiseed" / f"{experiment_name}_seed_{training_seed}.pt"
            )
            history = fit_model(
                model,
                loaders["train"],
                loaders["validation"],
                DEVICE,
                checkpoint_path,
                epochs=EPOCHS,
                learning_rate=LEARNING_RATE,
            )
            metrics = evaluate_model(model, loaders["test"], DEVICE, CLASS_NAMES)
            history.to_csv(
                ARTIFACT_DIR / f"history_{experiment_name}_seed_{training_seed}.csv",
                index=False,
            )
            new_result = pd.DataFrame(
                [
                    {
                        "seed": training_seed,
                        "input_mode": experiment_name,
                        "accuracy": metrics["accuracy"],
                        "balanced_accuracy": metrics["balanced_accuracy"],
                        "macro_f1": metrics["macro_f1"],
                        "test_loss": metrics["loss"],
                    }
                ]
            )
            multi_seed_results = pd.concat(
                (multi_seed_results, new_result), ignore_index=True
            )
            multi_seed_results = multi_seed_results.drop_duplicates(
                subset=["seed", "input_mode"], keep="last"
            ).sort_values(["seed", "input_mode"])
            multi_seed_results.to_csv(MULTI_SEED_RESULTS_PATH, index=False)
            completed_runs.add(run_key)

            del model, loaders
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

display(
    multi_seed_results.groupby("input_mode")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
    )
    .style.format(precision=4)
)

In [ ]:
from scipy.stats import ranksums, wilcoxon

PRIMARY_METRIC = "accuracy"
ALPHA = 0.05
BASELINE_MODE = "rgb"
MASK_MODE = label_input_mode("rgb_mask")

paired_scores = multi_seed_results.pivot(
    index="seed", columns="input_mode", values=PRIMARY_METRIC
).dropna(subset=[BASELINE_MODE, MASK_MODE])
if len(paired_scores) < len(MULTI_SEEDS):
    raise RuntimeError(
        f"Only {len(paired_scores)} complete seed pairs are available; "
        f"run all {len(MULTI_SEEDS)} pairs before inference."
    )

baseline_scores = paired_scores[BASELINE_MODE].to_numpy()
mask_scores = paired_scores[MASK_MODE].to_numpy()
paired_differences = mask_scores - baseline_scores

if np.allclose(paired_differences, 0):
    signed_rank_statistic, signed_rank_p = 0.0, 1.0
else:
    signed_rank_result = wilcoxon(
        mask_scores,
        baseline_scores,
        alternative="greater",
        zero_method="wilcox",
        method="auto",
    )
    signed_rank_statistic = signed_rank_result.statistic
    signed_rank_p = signed_rank_result.pvalue

rank_sum_result = ranksums(mask_scores, baseline_scores, alternative="greater")

rng = np.random.default_rng(SEED)
bootstrap_indices = rng.integers(
    0, len(paired_differences), size=(10000, len(paired_differences))
)
bootstrap_mean_differences = paired_differences[bootstrap_indices].mean(axis=1)
ci_low, ci_high = np.quantile(bootstrap_mean_differences, [0.025, 0.975])

statistical_results = pd.DataFrame(
    [
        {
            "test": "Paired Wilcoxon signed-rank (primary)",
            "alternative": f"{MASK_MODE} > {BASELINE_MODE}",
            "statistic": signed_rank_statistic,
            "p_value": signed_rank_p,
            "significant_at_0.05": signed_rank_p < ALPHA,
        },
        {
            "test": "Wilcoxon rank-sum (secondary)",
            "alternative": f"{MASK_MODE} > {BASELINE_MODE}",
            "statistic": rank_sum_result.statistic,
            "p_value": rank_sum_result.pvalue,
            "significant_at_0.05": rank_sum_result.pvalue < ALPHA,
        },
    ]
)
statistical_results.to_csv(
    ARTIFACT_DIR / f"multiseed_statistics_{MASK_VARIANT}.csv", index=False
)

effect_summary = pd.DataFrame(
    [
        {
            "paired_seeds": len(paired_scores),
            "rgb_mean_accuracy": baseline_scores.mean(),
            "mask_mean_accuracy": mask_scores.mean(),
            "mean_paired_improvement": paired_differences.mean(),
            "median_paired_improvement": np.median(paired_differences),
            "mean_improvement_95ci_low": ci_low,
            "mean_improvement_95ci_high": ci_high,
            "seeds_improved": int((paired_differences > 0).sum()),
        }
    ]
)
effect_summary.to_csv(
    ARTIFACT_DIR / f"multiseed_effect_{MASK_VARIANT}.csv", index=False
)
display(effect_summary.style.format(precision=4))
display(statistical_results.style.format({"statistic": "{:.4f}", "p_value": "{:.6f}"}))

direction_is_positive = paired_differences.mean() > 0
if signed_rank_p < ALPHA and direction_is_positive:
    print(
        "Conclusion: the reverse-mask fourth channel significantly improves "
        f"accuracy across paired training seeds (one-sided paired Wilcoxon "
        f"signed-rank p={signed_rank_p:.4g})."
    )
else:
    print(
        "Conclusion: this experiment does not establish a statistically "
        f"significant accuracy improvement (one-sided paired Wilcoxon "
        f"signed-rank p={signed_rank_p:.4g})."
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x_positions = np.array([0, 1])
for seed, row in paired_scores.iterrows():
    axes[0].plot(
        x_positions,
        [row[BASELINE_MODE], row[MASK_MODE]],
        marker="o",
        alpha=0.7,
        label=f"Seed {seed}",
    )
axes[0].set_xticks(x_positions, ["RGB", "RGB + reverse mask"])
axes[0].set(title="Paired accuracy by training seed", ylabel="Test accuracy")

bar_colors = np.where(paired_differences >= 0, "#238636", "#cf222e")
axes[1].bar(paired_scores.index.astype(str), paired_differences, color=bar_colors)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Accuracy change from reverse-mask channel",
    xlabel="Training seed",
    ylabel="Mask accuracy - RGB accuracy",
)
plt.tight_layout()
plt.savefig(
    PLOT_DIR / f"multiseed_accuracy_{MASK_VARIANT}.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## Interpretation

The primary comparison is `rgb` versus `rgb_mask_reverse_mask`. An improvement in the reverse-mask fourth-channel condition, together with failure of `masked_rgb_reverse_mask` to improve, supports the report's hypothesis that MaskCut is more useful as auxiliary spatial information than as a hard pixel filter.

These pseudo-masks are not tumor annotations. They should be described as unsupervised spatial priors, and the experiment evaluates tissue classification rather than clinical cancer diagnosis.